In [ ]:

!pip install scikit-learn matplotlib seaborn joblib tqdm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from joblib import dump, load
import os

In [ ]:
print('Downloading MNIST...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X = mnist.data.astype(np.float32)/255.0
y = mnist.target.astype(int)

In [ ]:
idx = np.random.RandomState(42).choice(len(X), 5000, replace=False)
X = X[idx]
y = y[idx]

In [ ]:
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)
print("MNIST sample shape:", X_scaled.shape)

MNIST sample shape: (5000, 784)


In [ ]:
pca3 = PCA(n_components=3)
X_pca3 = pca3.fit_transform(X_scaled)

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(X_pca3[mask,0], X_pca3[mask,1], X_pca3[mask,2], label=str(cls), s=20, alpha=0.6)
ax.set_title('MNIST projected to 3D PCA')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.legend()
plt.show()

In [ ]:
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(cum_var)+1), cum_var, marker='o')
plt.axhline(0.95, color='r', linestyle='--', label='95% variance')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('Explained Variance - MNIST')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

k = 162
pca_img = PCA(n_components=k)
Xk = pca_img.fit_transform(X_scaled)
X_recon = pca_img.inverse_transform(Xk)

In [ ]:
fig, axes = plt.subplots(2,8, figsize=(12,3))
inds = np.random.choice(len(X_scaled), 8, replace=False)
for i, idx in enumerate(inds):
    axes[0,i].imshow(X_scaled[idx].reshape(28,28), cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(X_recon[idx].reshape(28,28), cmap='gray'); axes[1,i].axis('off')
axes[0,0].set_title('Original')
axes[1,0].set_title(f'Reconstructed (k={k})')
plt.show()

In [ ]:

mask_normal = y == 1
X_normal = X_scaled[mask_normal][:2000]
mask_others = y != 1
X_outliers = X_scaled[mask_others][:200]

In [ ]:
pca_anom = PCA(n_components=50).fit(X_normal)
recon_normal = pca_anom.inverse_transform(pca_anom.transform(X_normal))
recon_out = pca_anom.inverse_transform(pca_anom.transform(X_outliers))
err_normal = np.mean((X_normal - recon_normal)**2, axis=1)
err_out = np.mean((X_outliers - recon_out)**2, axis=1)

In [ ]:
plt.hist(err_normal, bins=50, alpha=0.7, label='normal')
plt.hist(err_out, bins=50, alpha=0.7, label='outliers')
plt.legend()
plt.title('Reconstruction error - anomaly detection')
plt.show()

In [ ]:

pca_k = PCA(n_components=10).fit_transform(X_scaled)
kmeans = KMeans(n_clusters=10, random_state=42).fit(pca_k)
labels_k = kmeans.labels_

In [ ]:

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
for i in range(10):
    mask = labels_k == i
    ax.scatter(pca_k[mask,0], pca_k[mask,1], pca_k[mask,2], s=20, alpha=0.6)
ax.set_title('KMeans on PCA-reduced MNIST (10D->3D plot)')
plt.show()

In [ ]:

ipca = IncrementalPCA(n_components=50, batch_size=2000)
for i in range(0, X_scaled.shape[0], 2000):
    ipca.partial_fit(X_scaled[i:i+2000])
X_ipca = ipca.transform(X_scaled[:5000])
print('IncrementalPCA transform shape (sample):', X_ipca.shape)

IncrementalPCA transform shape (sample): (5000, 50)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=2000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(kernel='rbf', gamma='scale')
}

In [ ]:
results = []
for name, model in models.items():
    score_no_pca = cross_val_score(model, X_train, y_train, cv=3).mean()
    pipeline = Pipeline([('pca', PCA(n_components=50)), ('clf', model)])
    score_pca = cross_val_score(pipeline, X_train, y_train, cv=3).mean()
    results.append((name, score_no_pca, score_pca))

print(f"{'Model':<20} {'No PCA':<10} {'PCA(50)':<10}")
for r in results:
    print(f"{r[0]:<20} {r[1]:<10.4f} {r[2]:<10.4f}")